[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wur-bioinformatics/data-science-concepts/blob/main/notebooks/block4_exploration_2/tutorial_4.ipynb)

When working in Colab, first use *File → Save a copy in Drive*. Changes to the original notebook are not saved.

# Tutorial 4 - Exploratory Data Analysis II
This tutorial deals with various aspects of data analysis: regression, classification, model evaluation, and a small primer on deep learning. The code block below import all dependencies for this notebooks.

#### Import libraries

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

from sklearn import datasets, linear_model, metrics
from sklearn.preprocessing import PolynomialFeatures
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score

RANDOM_STATE = 42

# Location of the course datasets
DATA_URL = 'https://raw.githubusercontent.com/wur-bioinformatics/data-science-concepts/main/datasets/'

# 1 Regression (~45 minutes)

Regression analysis refers to the **prediction** of a particular real-valued outcome or **dependent variable** using a number of different independent variables or **features**. Some examples of regression in real life:

* Forecasting / predicting the amount of agricultural yield, based on factors like the amount of rainfall, pesticide usage, etc.
* A pharmaceutical company could use regression to predict things like shelf life and stability of drugs based on their properties.
* We'll be looking at a use-case below where regression can be used to predict the price of a house based on things like the size and number of rroms in the house, as well as more external factors relating to the neighborhood the house is in.

## 1.1 Linear Regression

The simplest form of regression is linear regression, in which we model a linear relationship between the **dependent variable** $y$ and a single **feature** $X$, with the formula:

$$y = coefficient * X + intercept$$

We explore this concept using some randomly generated data:

In [ ]:
X, y = datasets.make_regression(n_samples=100, n_features=1, noise=0, random_state=RANDOM_STATE)
sns.scatterplot(x=X[:, 0], y=y)

In [ ]:
reg = linear_model.LinearRegression()
reg.fit(X, y)
reg.coef_, reg.intercept_

We can use the fit coefficient and intercept to draw the line $y = coefficient * X + intercept$ through the actual data points.

In [ ]:
# specify a range of X values for which we want to evaluate the model
xs = np.arange(min(X[:,0]), max(X[:,0]) + 1)

# predict Y from X using linear regression coefficients: Y = aX + b
ys = reg.coef_ * xs + reg.intercept_

# plot data points and regression line
sns.scatterplot(x=X[:, 0], y=y)
plt.plot(xs, ys, color="black");

Once we have this line, we can use it to predict the value of $y$ for any value of $X$, by just plugging in this value with the coefficient and intercept into the formula. This is accomplished via the `predict` function below. We can then compare these predictions to the true values of $y$ and see how far the prediction is from the truth. We do this in three ways:

* Predicted vs. True - plots the true value on the x-axis and the predicted value on the y-axis. 

* Residuals - plots the predicted value on the x-axis and the difference between the true value and predicted value (called the **residual**) on the y-axis

* Residual counts - plots a histogram of the frequencies of different **residual** values.



In [ ]:
y_pred = reg.predict(X)
residuals = y - y_pred

fig, ax = plt.subplots(1, 3, figsize=(20,5))
ax[0].set_title("Predicted vs. True")
ax[0].set_ylabel("Predicted y")
sns.scatterplot(x=y, y=y_pred, ax=ax[0])

ax[1].set_title("Residuals")
ax[1].set_xlabel("Predicted y")
ax[1].set_ylabel("Difference")
ax[1].plot([min(y_pred), max(y_pred)], [0, 0], 'k--', alpha=0.7)
sns.scatterplot(x=y_pred, y=residuals, ax=ax[1])

ax[2].set_title("Residual counts")
sns.histplot(residuals, ax=ax[2], bins=20, kde=True);

Try changing the value of `noise` in the first cell that makes $X$ and $y$ above and refitting the linear regression.

> 1.1 How does this affect the linear regression line?

---
Answer:

---

> 1.2 How do the above three plots look when the relationship between $X$ and $y$ is perfectly linear? Hint: try setting noise to 0.

---
Answer:

---

#### Loading the Automobile dataset
Let's look at linear regression with some real data. The code block below loads the [Automobile dataset](https://archive.ics.uci.edu/dataset/10/automobile), which has a variety of numeric, categorical, and binary variables on cars from the 80's.


In [ ]:
# A copy of the automobile dataset from the UCI repository (https://archive.ics.uci.edu/dataset/10/automobile)
automobile_df = pd.read_csv(DATA_URL + 'automobile.csv')
# The dataset contains missing values, here we remove all rows that contain NAs
na_rows = automobile_df.isna().sum(axis=1) > 0
automobile_df = automobile_df.loc[~na_rows]
# For regression we are only interested in numbers
automobile_df = automobile_df.select_dtypes(include='number')
automobile_df

First, we'll take a look at price vs. all other features to see if any of them have a linear relationship.

In [ ]:
target = 'price'
# Plot pairwise relationships of all variables VS price (double clicking enlarges the plot)
sns.pairplot(automobile_df, x_vars=[c for c in automobile_df.columns if c != target], y_vars=[target])
# Plot a histogram of price
sns.pairplot(automobile_df, x_vars=[target], y_vars=[target])

> 1.3 Which features seem to have a linear relationship with price?

---
Answer:

---

> 1.4 Which features seem to not have any relationship with price?

---
Answer:

---

It looks like horsepower has a some relationship with price. We can try fitting a linear regression model to this. This models $$price = coefficient * horsepower + intercept$$ 

In [ ]:
X = automobile_df[["horsepower"]]
y = automobile_df["price"]

In [ ]:
reg = linear_model.LinearRegression()
reg.fit(X, y)

In [ ]:
reg.coef_, reg.intercept_

We can use the fit coefficient and intercept to draw the line $price = coefficient * horsepower + intercept$ through the actual data points.

In [ ]:
xs = np.arange(min(X.values[:,0]), max(X.values[:,0]) + 1)
ys = xs * reg.coef_ + reg.intercept_

sns.scatterplot(x=X["horsepower"], y=y)
plt.plot(xs, ys, color="black")

> 1.5 Do you think this line represents the underlying data well?

---
Answer:

---

Now let's _predict_ the value of price on the automobile dataset points, and compare to the real price values

In [ ]:
y_pred = reg.predict(X)
residuals = y - y_pred

fig, ax = plt.subplots(1, 3, figsize=(20,5))
ax[0].set_title("Predicted vs. True")
ax[0].set_ylabel("Predicted price")
sns.scatterplot(x=y, y=y_pred, ax=ax[0])

ax[1].set_title("Residuals")
ax[1].set_xlabel("Predicted price")
ax[1].set_ylabel("Difference")
ax[1].plot([min(y_pred), max(y_pred)], [0, 0], 'k--', alpha=0.7)
sns.scatterplot(x=y_pred, y=residuals.values, ax=ax[1])

ax[2].set_title("Residual counts")
sns.histplot(residuals, kde=True, ax=ax[2]);

> 1.6 Is the relationship between price and horsepower perfectly linear?

---
Answer:

---

## 1.2 Multiple Linear Regression

Now let's extend this concept to use all the features in the automobile dataset to predict price. All this does is extend the formula:
$$price = coefficient_1 * highway-mpg + coefficient_2 * city-mpg + ... + coefficient_{16} * normalized-losses + intercept$$ 

In [ ]:
X = automobile_df.drop("price", axis=1)
y = automobile_df["price"]

In [ ]:
reg = linear_model.LinearRegression()
reg.fit(X, y)

In [ ]:
print("Intercept:", reg.intercept_)
coefficients = pd.DataFrame(reg.coef_, X.columns, columns=['Coefficient'])
coefficients

These coefficients can be easily interpreted. For example, *holding all the other features fixed*, a 1 unit increase in the engine size is associated with a predicted increase in the price of 24.96 dollars.

> 1.7 Which features have a positive relationship with price (i.e. increasing the feature value increasing the price of the car)? And which ones have a negative relationship?

---
Answer:

---

> 1.8 Which features most effect (positively or negatively) the car price? Do these make intuitive sense?

---
Answer:

---

In [ ]:
# Plotting prediction and residuals for multiple linear regression
y_pred = reg.predict(X)
residuals = y - y_pred

fig, ax = plt.subplots(1, 3, figsize=(20,5))
# Predicted VS True plot
ax[0].set_title("Predicted vs. True")
ax[0].set_ylabel("Predicted price")
sns.scatterplot(x=y, y=y_pred, ax=ax[0])

# Residuals scatterplot
ax[1].set_title("Residuals")
ax[1].set_xlabel("Predicted price")
ax[1].set_ylabel("Difference")
ax[1].plot([min(y_pred), max(y_pred)], [0, 0], 'k--', alpha=0.7)
sns.scatterplot(x=y_pred, y=residuals.values, ax=ax[1])

# Residuals histogram
ax[2].set_title("Residual count")
sns.histplot(residuals, kde=True, ax=ax[2]);

> 1.9 Compare these three plots to the single linear regression plots above (pay attention to the scales!). Is the multiple linear regression a better fit than the single?

---
Answer:

---

> __*OPTIONAL*__: 1.10 Using features except highway-mpg (but including price), fit single and multiple linear regression models to predict the fuel efficiency on the highway of a particular car model.

---
Answer:

---

## 1.3 Nonlinear Regression

Let's take another look at the pairplot between price and the other features

In [ ]:
# Same code as above
target = 'price'
sns.pairplot(automobile_df, x_vars=[c for c in automobile_df.columns if c != target], y_vars=[target])
sns.pairplot(automobile_df, x_vars=[target], y_vars=[target])

> 1.11 Do any of the features have a relationship with price that is not linear?

---
Answer:

---

Let's further look into highway-mpg (fuel efficiency in miles per gallon while driving on the highway), which seems to have an interesting relationship with price:

In [ ]:
sns.scatterplot(data=automobile_df, x="highway-mpg", y="price");

We can fit a linear regression through it:

In [ ]:
X = automobile_df[["highway-mpg"]]
y = automobile_df["price"]

In [ ]:
reg = linear_model.LinearRegression()
reg.fit(X, y)

In [ ]:
y_pred = reg.predict(X)
residuals = y - y_pred

fig, ax = plt.subplots(1, 4, figsize=(20,4))
ax[0].set_title("highway-mpg vs price")
xs = np.arange(min(X.values[:,0]), max(X.values[:,0]) + 1)
ys = xs * reg.coef_ + reg.intercept_
sns.scatterplot(x=X["highway-mpg"], y=y, ax=ax[0])
ax[0].plot(xs, ys, color="black")

ax[1].set_title("Predicted vs. True")
ax[1].set_ylabel("Predicted price")
sns.scatterplot(x=y, y=y_pred, ax=ax[1])

ax[2].set_title("Residuals")
ax[2].set_xlabel("Predicted price")
ax[2].set_ylabel("Difference")
ax[2].plot([min(y_pred), max(y_pred)], [0, 0], 'k--', alpha=0.7)
sns.scatterplot(x=y_pred, y=residuals.values, ax=ax[2])


ax[3].set_title("Residual count")
sns.histplot(residuals, kde=True, ax=ax[3])

> 1.12 Does this seem like a good fit?

---
Answer:

---

Instead, let's fit a quadratic (degree 2) polynomial, i.e. 
$$price = coefficient_1 * highway\text-mpg + coefficient_2 * highway\text-mpg^2 + intercept$$

We'll use the PolynomialFeatures class in scikit-learn to convert $X$ into $[1, X, X^2]$

In [ ]:
poly_X = PolynomialFeatures(degree=2).fit_transform(X)

In [ ]:
X.shape, poly_X.shape

And then fit a multiple linear regression model to `poly_X`

In [ ]:
reg = linear_model.LinearRegression()
reg.fit(poly_X, y)

In [ ]:
print("Intercept:", reg.intercept_)
for name, coef in zip(["highway-mpg", "highway-mpg^2"], reg.coef_[1:]):
    print(f"{name}:", coef)

In [ ]:
y_pred = reg.predict(poly_X)
residuals = y - y_pred

fig, ax = plt.subplots(1, 4, figsize=(20,4))
ax[0].set_title("highway-mpg vs price")
xs = np.arange(min(X.values[:,0]), max(X.values[:,0]) + 1)
ys = reg.coef_[1] * xs + reg.coef_[2] * xs **2 + reg.intercept_
sns.scatterplot(x=X["highway-mpg"], y=y, ax=ax[0])
ax[0].plot(xs, ys, color="black")

ax[1].set_title("Predicted vs. True")
ax[1].set_ylabel("Predicted price")
sns.scatterplot(x=y, y=y_pred, ax=ax[1])

ax[2].set_title("Residuals")
ax[2].set_xlabel("Predicted price")
ax[2].set_ylabel("Difference")
ax[2].plot([min(y_pred), max(y_pred)], [0, 0], 'k--', alpha=0.7)
sns.scatterplot(x=y_pred, y=residuals.values, ax=ax[2])


ax[3].set_title("Residual count")
sns.histplot(residuals, kde=True, ax=ax[3])

> 1.13 Compare this to the linear fit above. 

---
Answer:

---

Just as we went from single linear regression to multiple linear regression, we can combine polynomial terms from  multiple features. One thing to note here is that we can also take into account interaction terms between different features. So the formula for the model below would be:

$$price = coefficient_1 * highway\text-mpg $$
$$+ coefficient_2 * horsepower $$
$$+ coefficient_3 * highway\text-mpg^2$$
$$+ coefficient_4 * horsepower * highway\text-mpg$$
$$+ coefficient_5 * horsepower^2$$
$$+ intercept $$


In [ ]:
X = automobile_df[["highway-mpg", "horsepower"]]
y = automobile_df["price"]

In [ ]:
poly = PolynomialFeatures(degree=2)
poly_X = poly.fit_transform(X)
X.shape, poly_X.shape

In [ ]:
reg = linear_model.LinearRegression()
reg.fit(poly_X, y)

In [ ]:
print("Intercept:", reg.intercept_)
for name, coef in zip(["highway-mpg", "horsepower", "highway-mpg^2", "highway-mpg * horsepower", "horsepower^2"], reg.coef_[1:]):
    print(f"{name}:", coef)

In [ ]:
y_pred = reg.predict(poly_X)
residuals = y - y_pred

fig, ax = plt.subplots(1, 3, figsize=(20,5))

ax[0].set_title("Predicted vs. True")
ax[0].set_ylabel("Predicted price")
sns.scatterplot(x=y, y=y_pred, ax=ax[0])

ax[1].set_title("Residuals")
ax[1].set_xlabel("Predicted price")
ax[1].set_ylabel("Difference")
ax[1].plot([min(y_pred), max(y_pred)], [0, 0], 'k--', alpha=0.7)
sns.scatterplot(x=y_pred, y=residuals.values, ax=ax[1])


ax[2].set_title("Residual count")
sns.histplot(residuals, kde=True, ax=ax[2])

> 1.14 What do you think of this fit?

---
Answer:

---

> __*OPTIONAL*__: 1.15 Can you do something similar for predicting the highway fuel effificiency (highway-mpg)? Make sure to look at the pairplots for an indication of nonlinear relationships

---
Answer:

---

# 2 Classification (~45 minutes)

In the Clustering notebook we explored datasets which have a natural grouping (e.g. species for Iris, digit for MNIST). While the purpose of clustering and other unsupervised methods is to look for and understand structure and patterns in your data, in this notebook we'll focus on actually predicting this grouping i.e. classifying or categorizing data points into a small number of groups.

Some examples of classification in the real world:
* Your email provider has an algorithm to "classify" emails as spam or not spam. 
* An OCR (automatic character recognition) algorithm can take an image of a hand-drawn character and decide which alphabet it represents.
* Some online newspapers can have algorithms to automatically place articles into different categories, like Politics, Sports, Literature etc.

There are a number of algorithms to perform classification, here we'll look at two popular ones.

### Simulated Data

To better understand how different algorithms work we'll make some randomly simulated datasets. These datasets will have points from two different classes, and will have two features each so that they are easier to plot.

In [ ]:
X_blobs, y_blobs = datasets.make_blobs(n_samples=500, n_features=2, centers=2, cluster_std=1.5, random_state=RANDOM_STATE)
X_blobs = pd.DataFrame(X_blobs, columns=["feature_1", "feature_2"])
                                
X_moons, y_moons = datasets.make_moons(n_samples=500, noise=0.1, random_state=RANDOM_STATE)
X_moons = pd.DataFrame(X_moons, columns=["feature_1", "feature_2"])
                                
X_circles, y_circles = datasets.make_circles(n_samples=500, noise=0.1, factor=.1, random_state=RANDOM_STATE)
X_circles = pd.DataFrame(X_circles, columns=["feature_1", "feature_2"])


fig, ax = plt.subplots(1, 3, figsize=(20,5))
sns.scatterplot(data=X_blobs, x="feature_1", y="feature_2", hue=y_blobs, ax=ax[0])
sns.scatterplot(data=X_moons, x="feature_1", y="feature_2", hue=y_moons, ax=ax[1])
sns.scatterplot(data=X_circles, x="feature_1", y="feature_2", hue=y_circles, ax=ax[2]);

> 2.1 Can you come up with a line or curve that can divide these points into the two colored groups for each dataset? Is this the same line in each case?

---
Answer:

---

Try changing the values of `cluster_std` in line 1 and `noise` in lines 4 and 7. 

> 2.2 How does adding (or decreasing) noise change the lines that you thought about in Question 1?

---
Answer:

---

Next, we'll train classification algorithms to come up with these lines or curves, called **decision boundaries**, for us.

The function `plot_decision_boundary` from the course package `data-science-concepts` plots the decision boundary of a classifier by coloring the data space according to the class predicted. What it does will be made clear when we use the function.

In [ ]:
# Install the course package with helper functions
%pip install -q "data-science-concepts==2026.*"
import data_science_concepts as dsc

## 2.1 $K$-Nearest Neighbors Classifier

Let's start with one of the simplest classifiers - the $k$-nearest neighbors classifier. As the name suggests, this classifier looks at the $k$ nearest points to a given point, checkes their classes, and uses a majority vote strategy to decide the class of the given point.

> 2.3 Pick two points in each of the three dataset plots above, and find their 5 nearest neighbors. What would the class of your chosen points be according to their nearest neighbors? Is this always correct?

---
Answer:

---

> 2.4 Increase the `noise` (and `cluster_std`) values, rerun the first cell, and repeat the above exercise. Are the predictions still correct?

---
Answer:

---

Let's do this with a library now instead of by eye:

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(20,5))
n_neighbors = 5

clf1 = KNeighborsClassifier(n_neighbors=n_neighbors)
clf1.fit(X_blobs.values, y_blobs)
dsc.plot_decision_boundary(X_blobs, y_blobs, ["feature_1", "feature_2"], clf1, ax=ax[0])

clf2 = KNeighborsClassifier(n_neighbors=n_neighbors)
clf2.fit(X_moons.values, y_moons)
dsc.plot_decision_boundary(X_moons, y_moons, ["feature_1", "feature_2"], clf2, ax=ax[1])

clf3 = KNeighborsClassifier(n_neighbors=n_neighbors)
clf3.fit(X_circles.values, y_circles)
dsc.plot_decision_boundary(X_circles, y_circles, ["feature_1", "feature_2"], clf3, ax=ax[2])

The color in these plots represents the predicted class (class 0 or class 1) as well as the confidence of that prediction (i.e. light blue means that the algorithm not as sure about the prediction of class 0 as darker blue). 

> 2.5 What can you say about regions in the space which are far away from all the data points, in the first and second plots?

---
Answer:

---

> 2.6 How do these decision boundaries change when you increase / decrease the noise in the datasets? (Rerun data generation cell with the new noise values, then run this cell again)

---
Answer:

---

> 2.7 How does the number of neighbors used effect the decision boundaries? (Change the `n_neighbors` values in line 2 above)

---
Answer:

---

## 2.2 Decision Tree

A decision tree classifier makes a flowchart-like structure where each node represents a certain split in the data based on one feature value. For example - one node could be the split represented by "feature_1 < 0.5" - this leads to two branches in the tree, the first containing points with feature_1 < 0.5 and the second containing points with feature_1 > 0.5. Its goal is to find feature splits that can distinguish between the different classes. 

> 2.8 Can you think of a feature split for the first dataset (blobs) that divides the points into the two classes?

---
Answer:

---

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(20,5))
max_depth = 5

clf1 = DecisionTreeClassifier(max_depth=max_depth)
clf1.fit(X_blobs.values, y_blobs)
dsc.plot_decision_boundary(X_blobs, y_blobs, ["feature_1", "feature_2"], clf1, ax=ax[0])

clf2 = DecisionTreeClassifier(max_depth=max_depth)
clf2.fit(X_moons.values, y_moons)
dsc.plot_decision_boundary(X_moons, y_moons, ["feature_1", "feature_2"], clf2, ax=ax[1])

clf3 = DecisionTreeClassifier(max_depth=max_depth)
clf3.fit(X_circles.values, y_circles)
dsc.plot_decision_boundary(X_circles, y_circles, ["feature_1", "feature_2"], clf3, ax=ax[2])

One parameter of the decision tree is `max_depth`, which represents the maximum allowed depth of the tree. A `max_depth` of 1 would mean the tree can have exactly one node with two branches coming out of it. While a `max_depth` of 2 would mean each of these two branches can also have a node with another two branches attached.

> 2.9 What is the maximum number of nodes for a tree of `max_depth` = 3?

---
Answer:

---

Let's take a look at the trees themselves:

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(40,20))
plot_tree(clf1, filled=True, ax=ax[0], feature_names=["feature_1", "feature_2"]);
plot_tree(clf2, filled=True, ax=ax[1], feature_names=["feature_1", "feature_2"]);
plot_tree(clf3, filled=True, ax=ax[2], feature_names=["feature_1", "feature_2"]);

Each node above is colored according to the majority class of the points inside it (i.e. a blue node has mostly points from class 0 while an orange node has a lot of points from class 1). The exact number of points in each class is written in the "value" part of the node. Each point is put through this flowchart-like decision process until it ends up at a leaf (a node with no branches). A well-trained decision tree would have "pure" leaves at the bottom of the tree, i.e. leaves with only one class of points in them.

> 2.10 Are these three trees well-trained?

---
Answer:

---

Experiment with increasing the `max_depth` value for each dataset. 
> 2.11 What do you think of the shape of the decision boundary for the moons and circles datasets at `max_depth`s of 2 and 5?

---
Answer:

---

> 2.12 Do all three trees grow in size when you increase the depth? Comment on their classification ability at different depths.

---
Answer:

---

To plot just one tree (for better visibility):

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10,10))
plot_tree(clf1, filled=True, ax=ax, feature_names=["feature_1", "feature_2"]);

## 2.3 Iris dataset

Let's apply our knowledge on real data, once again with the Iris dataset.

In [ ]:
iris_dataframe = pd.read_csv(DATA_URL + 'iris.csv')
iris_dataframe.head()

In [ ]:
feature_names = ["sepal length", "sepal length", "petal length", "petal width"]
X = iris_dataframe[feature_names]
y = iris_dataframe["species"]

In [ ]:
clf = DecisionTreeClassifier(max_depth=5)
clf.fit(X, y)

In [ ]:
plt.figure(figsize=(20,10))
plot_tree(clf, filled=True, feature_names=feature_names);

> 2.13 Is it possible to plot the decision boundary for this classifier as we did in the questions above?

---
Answer:

---

This is a major advantage of decision trees over nearest neighbors classifiers: even with a lot of features, you can understand which features contribute and how they contribute to the classification. 

# 3 Evaluation (~45 minutes)

We've been working on some simple machine learning predictors on simple datasets so far, as well as visualizing the results as best as we can. However, as the prediction algorithms and the datasets we use become more and more complex, one question that comes to mind is - will this predictor work on new data? Could the predictor be merely memorizing the data we give it, and therefore be unable to make good predictions on future samples, or samples that it hasn’t seen before? This is quite an important question and has led to a lot of headaches in past endeavours:

**Examples of overfitting**

To make sure our predictors actually do perform / generalize well on new, unseen data we *evaluate* them, as we'll see below

Continuing with the theme of starting out with simulated data, let's take a look at the simple generated *moons* dataset from the classification notebook again. 

Suppose instead of having all 500 points, you only had 400 points for **training** your classifier. And then, you're asked to give these extra 100 **test** points to your trained classifier and report its predictions. Since you know where the 100 points belong, you can check and see if the predictions are close to the truth or not.

If these test points look something like the ones below (in white) then probably your classifier can do a good job:

In [ ]:
X_moons, y_moons = datasets.make_moons(n_samples=500, noise=0.1, random_state=RANDOM_STATE)
train_indices = np.random.randint(0, len(y_moons), 400)
test_indices = [i for i in range(len(y_moons)) if i not in train_indices]

X_moons_train = pd.DataFrame(X_moons[train_indices], columns=["feature_1", "feature_2"])
X_moons_test = pd.DataFrame(X_moons[test_indices], columns=["feature_1", "feature_2"])
y_train = y_moons[train_indices]
y_test = y_moons[test_indices]

fig, ax = plt.subplots(figsize=(10,7))
sns.scatterplot(data=X_moons_train, x="feature_1", y="feature_2", hue=y_train, ax=ax)
sns.scatterplot(data=X_moons_test, x="feature_1", y="feature_2", color="white", 
                ax=ax, legend=False, edgecolor="black");

But what if, by chance, the left out test points are not so well-distributed across the space and instead looked like the white points below?

In [ ]:
train_indices = [i for i in range(len(y_moons)) if y_moons[i] == 0 or X_moons[i, 0] > 0.5]
test_indices = [i for i in range(len(y_moons)) if y_moons[i] == 1 and X_moons[i, 0] <= 0.5]

X_moons_train = pd.DataFrame(X_moons[train_indices], columns=["feature_1", "feature_2"])
X_moons_test = pd.DataFrame(X_moons[test_indices], columns=["feature_1", "feature_2"])
y_train = y_moons[train_indices]
y_test = y_moons[test_indices]

fig, ax = plt.subplots(figsize=(10,7))
sns.scatterplot(data=X_moons_train, x="feature_1", y="feature_2", hue=y_train, ax=ax)
sns.scatterplot(data=X_moons_test, x="feature_1", y="feature_2", color="white", 
                ax=ax, legend=False, edgecolor="black");


> 3.1 Which class do you think your trained classifier would predict for each of these test points? Hint: Think of the decision tree boundaries from the previous notebook. 

---
Answer:

---

## 3.1 Evaluating Classification Models

Instead of visually inspecting points and trying to guess what would happen, let's train some classifiers and see how they perform on this test set. We can measure this *performance* in various ways - such as using a **confusion matrix**, which compares the correctly predicted samples to the incorrectly predicted ones. There are various metrics defined on the confusion matrix, such as the accuracy, precision, sensitivity etc.

![Confusion Matrix 2 classes](https://2.bp.blogspot.com/-EvSXDotTOwc/XMfeOGZ-CVI/AAAAAAAAEiE/oePFfvhfOQM11dgRn9FkPxlegCXbgOF4QCLcBGAs/s1600/confusionMatrxiUpdated.jpg)

We'll start out with a simple, 3-depth decision tree:

In [ ]:
clf = DecisionTreeClassifier(max_depth=3)
clf.fit(X_moons_train, y_train)

And then make predictions on the test data

In [ ]:
y_pred = clf.predict(X_moons_test)

Then we plot the confusion matrix for these predictions, with the actual class as the rows and the predicted class as the columns

In [ ]:
mat = metrics.confusion_matrix(y_test, y_pred)
sns.heatmap(mat, square=True, annot=True, fmt='d', cbar=False)
plt.ylabel('Actual Class')
plt.xlabel('Predicted Class');

> 3.2 How many True Positives, True Negatives, False Positives, and False Negatives are there?

---
Answer:

---

> 3.3 Are these predictions what you expected from Question 29?

---
Answer:

---

The `scikit-learn` metrics module has a lot of functions to calculate various performance metrics based on a set of predicted values and a set of true values. Below, we calculate the accuracy, precision, and recall (sensitivity).

In [ ]:
metrics.accuracy_score(y_test, y_pred)

In [ ]:
metrics.precision_score(y_test, y_pred)

In [ ]:
metrics.recall_score(y_test, y_pred)

The confusion matrix concept extends very naturally to cases like the Iris dataset where we have three classes: 

![Multiclass confusion matrix](https://raw.githubusercontent.com/wur-bioinformatics/data-science-concepts/main/notebooks/block4_exploration_2/multiclass_confusion_matrix.png)

In [ ]:
iris_dataset = datasets.load_iris()
iris_dataframe = pd.DataFrame(iris_dataset.data,
                              columns=iris_dataset.feature_names)
iris_dataframe["species"] = iris_dataset.target
iris_dataframe.head()

`scikit-learn` also has a function to randomly divide our dataset into **train** and **test** sets, as below. The `train_size` parameter says that 70% of the data will end up in the training set and 30% in the test set.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(iris_dataframe[iris_dataframe.columns[:-1]], 
                                                    iris_dataframe["species"], 
                                                    train_size=0.7, random_state=RANDOM_STATE)

In [ ]:
clf = DecisionTreeClassifier(max_depth=3)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
mat = metrics.confusion_matrix(y_test, y_pred)
sns.heatmap(mat.T, square=True, annot=True, fmt='d', cbar=False)
plt.xlabel('Actual Class')
plt.ylabel('Predicted Class');

In [ ]:
metrics.accuracy_score(y_test, y_pred)

> 3.4 What is the accuracy, precision, and recall here? What does it tell you about the predictor?

---
Answer:

---

> 3.5 Change the `train_size` parameter to lower values and re-evaluate. Is the new predictor still able to generalize to the test data?

---
Answer:

---

The Iris dataset in general is quite an easy one to classify even with a small training set, let's look at the more complicated MNIST dataset instead:

In [ ]:
# Loading data
mnist_dataset = datasets.load_digits()
mnist_dataframe = pd.DataFrame(mnist_dataset.data)
mnist_dataframe["digit"] = mnist_dataset.target
X = mnist_dataframe[mnist_dataframe.columns[:-1]]
y = mnist_dataframe["digit"]

# Splitting train and test tests
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, 
                                                    random_state=RANDOM_STATE)

# Training
clf = DecisionTreeClassifier(max_depth=5)
clf.fit(X_train, y_train)

# Predicting
y_pred = clf.predict(X_test)

# Confusion matrix
mat = metrics.confusion_matrix(y_test, clf.predict(X_test))
plt.figure(figsize=(10,7))
sns.heatmap(mat.T, square=True, annot=True, fmt='d', cbar=False,
            xticklabels=mnist_dataset.target_names,
            yticklabels=mnist_dataset.target_names)
plt.xlabel('true label')
plt.ylabel('predicted label')

# Accuracy
print(f"Accuracy: {metrics.accuracy_score(y_test, y_pred)}")

> 3.6 Based on what you saw in the clustering notebook, do these misclassifications make sense?

---
Answer:

---

## 3.2 Evaluating Regression Models

The metrics we used above, accuracy, precision, recall etc., can't be used to evaluate regression. Instead we need metrics which take into account how close the predicted value is to the real one. One such measure is the Root Mean Squared Error (RMSE), with the formula:

$$\sqrt{\frac{\sum_{n=1}^{N}{(y_t - y_p)^2}}{N}}$$

i.e. for each test point, it calculates the square of the difference between the true value $y_t$ and the predicted value $y_p$, sums them up for all test points, divides by the number of test points, and takes the square root. Because of the square and the root, RMSE is in the same scale as the data, making it easier to interpret, as we'll see below with the Boston dataset.

In [ ]:
# Here we use the automobile dataset again, if this is not loaded in your environment check this notebook's section on linear regression
X = automobile_df.drop("price", axis=1)
y = automobile_df["price"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, random_state=RANDOM_STATE)

In [ ]:
reg = linear_model.LinearRegression()
reg.fit(X_train, y_train)

As before, we have the three plots we use to look at our regression fits, except now the test points are not used for training and are plotted in a different color.

In [ ]:
y_pred = reg.predict(X_test)
y_pred_train = reg.predict(X_train)
residuals_train = y_train - y_pred_train
residuals_test = y_test - y_pred

fig, ax = plt.subplots(1, 3, figsize=(20,5))
ax[0].set_title("Predicted vs. True")
ax[0].set_ylabel("Predicted price")
sns.scatterplot(x=y_train, y=y_pred_train, ax=ax[0])
sns.scatterplot(x=y_test, y=y_pred, ax=ax[0])

ax[1].set_title("Residuals")
ax[1].set_xlabel("Predicted price")
ax[1].set_ylabel("Difference")
ax[1].plot([min(y_pred_train), max(y_pred_train)], [0, 0], 'k--', alpha=0.7)
sns.scatterplot(x=y_pred_train, y=residuals_train.values, ax=ax[1])
sns.scatterplot(x=y_pred, y=residuals_test.values, ax=ax[1])


ax[2].set_title("Residual count")
sns.histplot(residuals_train, kde=True, ax=ax[2])
sns.histplot(residuals_test, kde=True, color='orange', ax=ax[2]);

In [ ]:
print("RMSE: ", np.sqrt(metrics.mean_squared_error(y_test, y_pred)))

This indicates that, on average, our predictions are $2831.30 away from the truth.

> 3.7 Run `automobile_df.describe()` in a new cell and comment on whether 2831.30 is a reasonable error to have given the distribution of price values. 

---
Answer:

---

In [ ]:
automobile_df.describe()

## 3.3 Cross validation

The very first example with the moons dataset showed that test results can depend on properties of the test set. In such cases, having a single measure of performance based on a single test set may still be misleading. To make up for this we can use multiple different test (and train) sets and show how well the predictor performs across all of them.

![K fold cross validation](https://upload.wikimedia.org/wikipedia/commons/thumb/b/b5/K-fold_cross_validation_EN.svg/521px-K-fold_cross_validation_EN.svg.png)

The `cross_val_score` function in `scikit-learn` does this for us - it takes as input a predictor, the data, and the labels and performs $k$-fold cross-validation as described above. It then returns the requested performance metric values for each of the $k$ folds.

In [ ]:
# Iris Dataset Classification on species
clf = DecisionTreeClassifier(max_depth=3)
X = iris_dataframe[iris_dataframe.columns[:-1]]
y = iris_dataframe[iris_dataframe.columns[-1]]

# Accuracy values for each fold
cross_val_score(clf, X, y, cv=4, scoring="accuracy")

In [ ]:
# MNIST Dataset Classification on digits
clf = DecisionTreeClassifier(max_depth=5)
X = mnist_dataframe[mnist_dataframe.columns[:-1]]
y = mnist_dataframe[mnist_dataframe.columns[-1]]

# Accuracy values for each fold
cross_val_score(clf, X, y, cv=4, scoring="accuracy")

In [ ]:
# Boston Dataset Regression on MEDV
reg = linear_model.LinearRegression()
X = automobile_df.drop("price", axis=1)
y = automobile_df["price"]

# RMSE values for each fold
-1 * cross_val_score(reg, X, y, cv=4, scoring="neg_root_mean_squared_error")

> 3.8 Play around with the parameters of each of the predictors above to try and see if you can obtain better cross-validation performance.

---
Answer:

---

# 4: Deep Learning (~30 minutes)

The rapid increase in computational power in modern computers has led to a resurgence of another popular set of machine learning algorithms falling under the umbrella of *deep learning* or *neural networks*. Many deep learning algorithms are based on the underlying principle of interconnected nodes (or *neurons*), arranged in *layers*, with *weight*ed connections. These weights are *learned* using training data in such a way that the final outcome (which is usually some kind of combination of the weights of the last layer of nodes) best describes the input classification or regression problem.

![neural network](https://miro.medium.com/max/791/1*_Ueb1eX--nkvmHKX_UzyAA.jpeg)

Unlike the previous notebooks, you'll be using an interactive website for playing around with neural networks rather than coding them yourselves. You can find this website here:

> # [Tensorflow Playground](https://playground.tensorflow.org/)

### Perceptron

To start off, let's look at a single node connected to the inputs. This is called a **perceptron**.

![perceptron](https://miro.medium.com/max/2870/1*n6sJ4yZQzwKL9wnF5wnVNg.png)

Here, $x1$, $x2$, $x3$.. are inputs (petal width, petal length,... for example), and $y$ is the predicted output (eg. Species). The circle labeled "Step Function" is where an *Activation function* is applied. This function converts the input that a neuron receives (which is the weighted sum of all its inputs) into an output value that it can pass on. 


The Tensorflow playground website uses a set of simulated datasets. We'll start with the simple two blob dataset that we used before. This has two inputs ($X1$ and $X2$), and two classes (orange and blue)

![blobs](https://storage.googleapis.com/gweb-cloudblog-publish/images/neural-networks-17h634.max-400x400.PNG)

To train a single perceptron on this dataset:
1. Select this dataset from the **Data** panel on the left and make the training percentage 70% (to use 70% of the data as the training set and 30% as the test set)
2. In The **Features** panel, make sure only $X1$ and $X2$ are selected.
3. Remove all the hidden layers in the **Hidden Layers** panel (click the '-' button on top)
4. Check the "Show Test Data" box
5. And finally, click the Play arrow on the top to train the model.

Networks will train indefinitely so just click pause when you think the boundary looks reasonable. 

> 4.1 Try increasing the noise in the dataset and see how that effects the boundary. 

---
Answer:

---

Try each of the different datasets in the **Data** panel.
> 4.2  Which ones can the perceptron handle?

---
Answer:

---

### Multilayer perceptrons

To get to the network in the first image, we'll need at least one hidden layer of neurons.
Add one hidden layer with 4 neurons and train the network for each of the four datasets. Make sure the "Activation" value is set to something other than "Linear".

> 4.3 Which of the four datasets can this network correctly model?

---
Answer:

---

#### Activation Functions

The activation function defines the output of a neuron (y-axis) given an input (x-axis). These are the functions defined in the Tensorflow Playground.

![activation functions](https://raw.githubusercontent.com/wur-bioinformatics/data-science-concepts/main/notebooks/block4_exploration_2/activation.png)

_Source: https://www.kdnuggets.com/wp-content/uploads/activation.png_

Make a neural network with 2 hidden layers each with up to 4 neurons in them and experiment with using different activation functions for each of the datasets. Also take a look at the curve in the **Output** panel which plots the training error as training proceeds.

> 4.4 Try to find a model that can fit the spiral dataset (the fourth one in the dataset list) just by adding/removing hidden layers and neurons, and changing the activation function.

---
Answer:

---

> 4.5 Now try fitting the fourth dataset by adding polynomial features such as $X1^2$, $X1X2$, or even $sin(X1)$

---
Answer:

---

You can see what each individual neuron is learning about the space by hovering over it.

## Beyond classification and regression

Deep learning algorithms are often used in complex problems which are extensions of the classification and regression definitions that you're used to. These include data compression and generative models. 

We'll look at a visual example of generative adversarial network (GAN) in [GANLAB website](https://poloclub.github.io/ganlab/). In this example, a GAN is used to generate a shape that looks similar to a set of input shapes when provided with random noise. 

Some real-life usecases of GANs:

* [Generative art](https://deepart.io/latest/) 
* [Generating realistic architecture images from a simple layout](https://affinelayer.com/pix2pix/)
* [Generating newspaper articles from a given sentence](https://openai.com/blog/better-language-models/)
